In [1]:
import scipy
import sys
from scipy import io
import numpy as np
import numpy.matlib
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import seaborn as sns
import pandas as pd
from numpy import matlib
import seaborn as sns
import math
import os
import scipy.optimize
import scipy.io as sio
from scipy import stats
import scipy.ndimage

In [2]:
##################### Experiment specifics
cutoff = None

expt_labels = ['bFB66_tun_pad', 'bFB69_LB_pad','bFB66_LB_pad']
pert=r'0.5 $\mu$g/mL Tunicamycin added'

linestyles = ['-.', '-', '--', '.']
out_path = '/Users/barber.527/Documents/GitHub/Rojas_lab_drafts/outputs/compiled_data/growth_rate_plots/'
in_path = '/Users/barber.527/Documents/GitHub/Rojas_lab_drafts/outputs/compiled_data/'

plot_labels = []

# loading the experimental parameters
for expt_label in expt_labels:
    temp_path = in_path + expt_label + '_condition_parameters.pkl'
    with open(temp_path, 'rb') as input:
        expt_vals = pickle.load(input)
    plot_labels.append(expt_vals['Celltype'])
vars = expt_vals['vars']  # This part should be conserved across all compiled experiments
normed = ['_normed', '']
# ylabels = {'sgr_l': r'$\frac{1}{L}\frac{dL}{dt}$', 'sgr_sa': r'$\frac{1}{S}\frac{dS}{dt}$', 'width':'Width'}
ylabels = {'sgr_l': r'Growth rate $\lambda_l$', 'sgr_sa': r'Growth rate $\lambda_S$', 'width': 'Width'}
yunits = {'sgr_l': r' $(h^{-1})$', 'sgr_sa': r' ($h^{-1}$)', 'width': r' ($\mu$m)'}
sel_vars = ['sgr_l', 'width']

Now we import our data and test significance across all variables listed

In [21]:
norm=''
time_testers=[200/60.0,63] # time in minutes for test to take place
for time_tester in time_testers:
    print('Time =', time_tester)
    for var in sel_vars:
        xvs=[]
        yvs=[]
        fig = plt.figure(figsize=[10, 6])
        for ind in range(len(expt_labels)):
            expt=expt_labels[ind]
            xv = np.load(in_path + expt + '_time.npy')
            if not cutoff is None:
                temp_vals = np.nonzero(xv / 60 < cutoff)[0][-1]
            else:
                temp_vals = len(xv) - 1
            xvs.append(xv[:temp_vals])
            yvs.append(np.load(in_path + expt + '_' + var + norm + '.npy')[:, :temp_vals])
        
        ind_vals=[np.nonzero(xvs[ind]==time_tester*60.0)[0][0] for ind in range(len(xvs))]
        for ind in range(len(ind_vals)):
            for ind1 in np.arange(ind+1,len(ind_vals)):
                print('Comparing conditions: ', expt_labels[ind],expt_labels[ind1])
                print('Variable = ',var)
                # print(xvs[ind][ind_vals[ind]]/60.0,xvs[ind1][ind_vals[ind1]]/60.0)
                temp_yv1=yvs[ind][:,ind_vals[ind]]
                temp_yv1=temp_yv1[np.nonzero(~np.isnan(temp_yv1))]
                temp_yv2=yvs[ind1][:,ind_vals[ind1]]
                temp_yv2=temp_yv2[np.nonzero(~np.isnan(temp_yv2))]
                print(scipy.stats.ttest_ind(temp_yv1,temp_yv2))

Time = 3.3333333333333335
Comparing conditions:  bFB66_tun_pad bFB69_LB_pad
Variable =  sgr_l
Ttest_indResult(statistic=-2.3105292350204203, pvalue=0.024313662281271395)
Comparing conditions:  bFB66_tun_pad bFB66_LB_pad
Variable =  sgr_l
Ttest_indResult(statistic=-13.631538793838384, pvalue=3.563899257655108e-20)
Comparing conditions:  bFB69_LB_pad bFB66_LB_pad
Variable =  sgr_l
Ttest_indResult(statistic=-11.41343729981447, pvalue=2.990743634602027e-19)
Comparing conditions:  bFB66_tun_pad bFB69_LB_pad
Variable =  width
Ttest_indResult(statistic=2.59383698969138, pvalue=0.011261153491591971)
Comparing conditions:  bFB66_tun_pad bFB66_LB_pad
Variable =  width
Ttest_indResult(statistic=-0.8834107030148528, pvalue=0.3785181608747765)
Comparing conditions:  bFB69_LB_pad bFB66_LB_pad
Variable =  width
Ttest_indResult(statistic=-4.913237646651379, pvalue=1.988350439914866e-06)
Time = 63
Comparing conditions:  bFB66_tun_pad bFB69_LB_pad
Variable =  sgr_l
Ttest_indResult(statistic=-5.133238314